# Step 6 · Pair-level disease evidence handoff

Step 4 screens the reproducible, capped PubMed sample for each configured pathogen–host pair using Jev's typed yes/no question. This notebook resumes from that completed decision ledger, verifies full-run coverage, and writes normalized pair decisions linked to their PMIDs and screened abstracts. It does not repeat model calls.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step, staging_root_for_config
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('06_handoff_disease_evidence', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-03 16:41 INFO Notebook session started at 2026-11-03T16:41:36.811699+00:00
2026-11-03 16:41 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 16:41 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/06_handoff_disease_evidence.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/06_handoff_disease_evidence.log


## Step 1 · Set up the run ledger and inspect literature inputs

In [2]:
import json
import pandas as pd, yaml
from IPython.display import display
from pathogen_atlas_v4.decision_identity import stable_hash
from pathogen_atlas_v4.decision_runner import decision_cache_key_for_task
from pathogen_atlas_v4.evidence import normalize_pair_gate
from pathogen_atlas_v4.pathogen_screening import build_pathogen_screen_tasks
from pathogen_atlas_v4.pubmed_client import literature_cache_key
from pathogen_atlas_v4.runs import resolve_run

CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = str(CONFIG['run']['run_id'])
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_RECORD = json.loads((RUN_DIR / 'resolved_config.json').read_text(encoding='utf-8'))
CONFIG = RUN_RECORD['source_config']
RUN_MODE = str(CONFIG['run'].get('selection_mode', '')).casefold()
IS_FULL_RUN = RUN_MODE == 'full'
STAGING_TAXONOMY_ROOT = staging_root_for_config(ROOT, CONFIG) / 'taxonomy'
RUN_TAXONOMY_ROOT = RUN_DIR / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy is missing; run notebooks 00 and 01 first.')
PATHOGENS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'hosts.parquet')
resolved = resolve_run(
    CONFIG,
    known_host_ids=set(HOSTS.host_id.astype(str)),
    known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)),
    host_group_ids={k: g.host_id.astype(str).tolist() for k, g in HOSTS.groupby('host_group')},
    pathogen_group_ids={k: g.pathogen_id.astype(str).tolist() for k, g in PATHOGENS.groupby('pathogen_group')},
)
CACHE_KEY = literature_cache_key(
    query_version=CONFIG['acquisition']['query_strategy_version'],
    pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
    taxonomy_snapshot=STAGING_TAXONOMY_ROOT.parent.name,
    host_expansion_hash=CONFIG['acquisition'].get('host_expansion_hash', ''),
)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / CACHE_KEY
PAIR_SEARCHES_PATH = CACHE_DIR / 'tables/pair_searches.parquet'
MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
SCREEN_LEDGER_PATH = ROOT / 'data/v4/cache/decisions/pathogen_screen_ledger.parquet'
PAIR_SELECTION_PATH = RUN_DIR / 'pathogen_host_selection.parquet'
LEDGER_PATH = RUN_DIR / 'decision_ledger.parquet'
EVIDENCE_PATH = RUN_DIR / 'evidence/pathogen_host_evidence.parquet'

required_inputs = [PAIR_SEARCHES_PATH, MATCHES_PATH, ARTICLES_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs and IS_FULL_RUN:
    raise FileNotFoundError('Full-mode Step 6 requires completed Steps 3 and 4 inputs: ' + ', '.join(missing_inputs))
if missing_inputs:
    raise FileNotFoundError('Pair-aware Steps 3 and 4 inputs are missing: ' + ', '.join(missing_inputs))

pair_searches = pd.read_parquet(PAIR_SEARCHES_PATH)
sample_matches = pd.read_parquet(MATCHES_PATH)
articles = pd.read_parquet(ARTICLES_PATH)
selected_pair_keys = set((str(p), str(h)) for p, h in resolved.pairs)
for frame_name, frame in [('pair searches', pair_searches), ('sample matches', sample_matches)]:
    if not frame.empty:
        pair_index = pd.MultiIndex.from_frame(frame[['pathogen_id', 'host_id']].astype(str))
        if frame_name == 'pair searches':
            pair_searches = frame.loc[pair_index.isin(selected_pair_keys)].copy()
        else:
            sample_matches = frame.loc[pair_index.isin(selected_pair_keys)].copy()

if IS_FULL_RUN:
    acquired_pair_keys = set(map(tuple, pair_searches.loc[
        pair_searches.status.astype(str).eq('completed'), ['pathogen_id', 'host_id']
    ].astype(str).drop_duplicates().to_numpy()))
    if selected_pair_keys - acquired_pair_keys:
        raise RuntimeError(f'Step 3 is incomplete: {len(selected_pair_keys - acquired_pair_keys):,} pairs lack a completed PubMed search. Rerun notebook 03.')
    searched_pair_keys = set(map(tuple, pair_searches.loc[
        pd.to_numeric(pair_searches.search_count, errors='coerce').fillna(0).gt(0), ['pathogen_id', 'host_id']
    ].astype(str).drop_duplicates().to_numpy()))
    prepared_pair_keys = set(map(tuple, sample_matches[['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy())) if not sample_matches.empty else set()
    if searched_pair_keys - prepared_pair_keys:
        raise RuntimeError(f'Step 4 sample preparation is incomplete for {len(searched_pair_keys - prepared_pair_keys):,} PubMed-hit pairs. Rerun notebook 03.')

print('Run:', RUN_ID, '| mode:', RUN_MODE, '| candidate pairs:', f'{len(resolved.pairs):,}')
print('Pair searches:', f'{len(pair_searches):,}', '| sampled pair records:', f'{len(sample_matches):,}')
print('Sampled abstracts available:', f'{int(sample_matches.abstract_available.fillna(False).sum()):,}')
print('Pair-aware cache:', CACHE_DIR)
LOG.info('Step 6 inputs: mode=%s candidate_pairs=%d searches=%d samples=%d', RUN_MODE, len(resolved.pairs), len(pair_searches), len(sample_matches))

2026-11-03 16:41 INFO Step 6 inputs: mode=full candidate_pairs=57151 searches=57151 samples=40028


Run: human-parasites-2026-11-03 | mode: full | candidate pairs: 57,151
Pair searches: 57,151 | sampled pair records: 40,028
Sampled abstracts available: 38,015
Pair-aware cache: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961


## Step 2 · Load pair-aware samples and verify full-run acquisition

In [3]:
expected_sample_pairs = set(map(tuple, pair_searches.loc[
    pd.to_numeric(pair_searches.search_count, errors='coerce').fillna(0).gt(0), ['pathogen_id', 'host_id']
].astype(str).drop_duplicates().to_numpy()))
abstract_sample_count = int(sample_matches.abstract_available.fillna(False).astype(bool).sum()) if 'abstract_available' in sample_matches else 0
print('PubMed-hit pairs with prepared samples:', f'{len(expected_sample_pairs):,}')
print('Sample records:', f'{len(sample_matches):,}', '| abstracts available:', f'{abstract_sample_count:,}')
print('Model-screen tasks are supplied by the Step 4 ledger; Step 6 makes no new provider calls.')
LOG.info('Step 6 preflight: pubmed_hit_pairs=%d sample_rows=%d abstracts=%d', len(expected_sample_pairs), len(sample_matches), abstract_sample_count)

2026-11-03 16:41 INFO Step 6 preflight: pubmed_hit_pairs=3777 sample_rows=40028 abstracts=38015


PubMed-hit pairs with prepared samples: 3,777
Sample records: 40,028 | abstracts available: 38,015
Model-screen tasks are supplied by the Step 4 ledger; Step 6 makes no new provider calls.


## Step 3 · Validate the Jev pair-decision record shape

In [4]:
# Synthetic Jev schema example only; it is not appended to the release ledger.
example = normalize_pair_gate(pathogen_id='bacteria:example', host_id='ncbi:9913', pmid='12345678',
    outcome='supported', confidence=None, evidence_text=None,
    title_hash='example-title-hash', abstract_hash='example-text-hash', run_id='schema-example',
    provenance_status='system1_jev', p_yes=0.93)
display(example)
LOG.info('Validated Jev supported decision schema with PMID, abstract hash, and p_yes')

{'pathogen_id': 'bacteria:example',
 'host_id': 'ncbi:9913',
 'relationship_context': 'disease',
 'pmid': '12345678',
 'title_hash': 'example-title-hash',
 'abstract_hash': 'example-text-hash',
 'evidence_span': None,
 'evidence_span_type': 'no_text_generated',
 'decision': 'supported',
 'confidence': None,
 'p_yes': 0.93,
 'provenance_status': 'system1_jev',
 'run_id': 'schema-example'}

2026-11-03 16:41 INFO Validated Jev supported decision schema with PMID, abstract hash, and p_yes


## Step 4 · Rebuild the screened task identities

In [5]:
PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
TASKS = build_pathogen_screen_tasks(
    sample_matches, articles, PATHOGENS, HOSTS, PROMPTS,
    run_id=RUN_ID,
    run_config_version=str(CONFIG['run']['config_version']),
    model_version=str(CONFIG['decisions']['model_version']),
)
TASK_BY_CACHE_KEY = {decision_cache_key_for_task(task): task for task in TASKS}
EXPECTED_CACHE_KEYS = set(TASK_BY_CACHE_KEY)
print('Expected completed abstract decisions:', f'{len(EXPECTED_CACHE_KEYS):,}')
print('Source screen ledger:', SCREEN_LEDGER_PATH)
print('Run decision ledger:', LEDGER_PATH)
print('Run evidence table:', EVIDENCE_PATH)
LOG.info('Rebuilt %d Step 4 task identities for Step 6 coverage check', len(EXPECTED_CACHE_KEYS))

2026-11-03 16:41 INFO Rebuilt 38015 Step 4 task identities for Step 6 coverage check


Expected completed abstract decisions: 38,015
Source screen ledger: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/decisions/pathogen_screen_ledger.parquet
Run decision ledger: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/decision_ledger.parquet
Run evidence table: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/evidence/pathogen_host_evidence.parquet


## Step 5 · Materialize the completed disease decisions

In [6]:
if SCREEN_LEDGER_PATH.exists():
    screen_ledger = pd.read_parquet(SCREEN_LEDGER_PATH)
else:
    screen_ledger = pd.DataFrame()

if screen_ledger.empty:
    relevant = pd.DataFrame()
else:
    relevant = screen_ledger.loc[screen_ledger.cache_key.astype(str).isin(EXPECTED_CACHE_KEYS)].copy()
    relevant = relevant.drop_duplicates('cache_key', keep='last')
completed = relevant.loc[relevant.status.astype(str).eq('completed')].copy() if not relevant.empty else relevant
completed_keys = set(completed.cache_key.astype(str)) if not completed.empty else set()
missing_keys = EXPECTED_CACHE_KEYS - completed_keys
if IS_FULL_RUN and missing_keys:
    raise RuntimeError(
        f'Full-mode Step 6 is blocked: {len(missing_keys):,} of {len(EXPECTED_CACHE_KEYS):,} '
        'sampled abstract decisions are not completed. Finish notebook 04 screening, then rerun this notebook.'
    )

normalized_rows = []
for record in completed.to_dict('records'):
    cache_key = str(record['cache_key'])
    task = TASK_BY_CACHE_KEY[cache_key]
    answer = str(record.get('answer', '')).strip().casefold()
    if answer not in {'yes', 'no'}:
        raise ValueError(f'Completed screen decision for PMID {task["pmid"]} has invalid answer {answer!r}')
    outcome = 'supported' if answer == 'yes' else 'unsupported'
    evidence_span_value = record.get('evidence_span')
    evidence_span = (
        '' if answer != 'yes' or evidence_span_value is None or pd.isna(evidence_span_value)
        else str(evidence_span_value).strip()
    )
    p_yes_value = record.get('p_yes')
    p_yes = None if pd.isna(p_yes_value) else float(p_yes_value)
    if answer == 'yes' and evidence_span and evidence_span not in str(task.get('abstract_text', '')):
        raise ValueError(f'Positive screen evidence for PMID {task["pmid"]} is not an exact quote from its abstract')
    confidence_value = record.get('confidence')
    confidence = (
        None
        if p_yes is not None or confidence_value is None or pd.isna(confidence_value)
        else float(confidence_value)
    )
    gate_row = normalize_pair_gate(
        pathogen_id=str(task['pathogen_id']), host_id=str(task['host_id']), pmid=str(task['pmid']),
        outcome=outcome, confidence=confidence, evidence_text=evidence_span,
        title_hash=stable_hash(str(task.get('title_text', ''))),
        abstract_hash=stable_hash(str(task.get('abstract_text', ''))), run_id=RUN_ID,
        relationship_context=str(task.get('relationship_context', 'disease')),
        provenance_status='system1_jev' if p_yes is not None and not evidence_span else 'v4_native',
        p_yes=p_yes,
    )
    gate_row.update({
        'scope': task['scope'], 'decision_kind': task['decision_kind'], 'answer': answer,
        'status': 'completed', 'attempts': record.get('attempts', 0), 'error': None,
        'model_version': task['model_version'], 'prompt_version': task['prompt_version'],
        'feature_version': task['feature_version'], 'question_version': task['question_version'],
        'profile_version': task['profile_version'], 'run_config_version': task['run_config_version'],
        'query_version': task['query_version'], 'query_hash': task.get('query_hash'),
        'input_hash': task['input_hash'], 'source_cache_key': cache_key,
    })
    normalized_rows.append(gate_row)

DECISION_COLUMNS = [
    'scope', 'decision_kind', 'pathogen_id', 'host_id', 'relationship_context', 'pmid',
    'title_hash', 'abstract_hash', 'evidence_span', 'evidence_span_type', 'decision', 'answer', 'confidence', 'p_yes',
    'provenance_status', 'run_id', 'model_version', 'prompt_version', 'feature_version',
    'question_version', 'profile_version', 'run_config_version', 'query_version', 'query_hash',
    'input_hash', 'source_cache_key', 'status', 'attempts', 'error',
]
DECISIONS = pd.DataFrame(normalized_rows, columns=DECISION_COLUMNS)
EVIDENCE_COLUMNS = [
    'pathogen_id', 'host_id', 'relationship_context', 'pmid', 'title_hash', 'abstract_hash',
    'evidence_span', 'evidence_span_type', 'decision', 'confidence', 'p_yes', 'provenance_status', 'run_id',
]
EVIDENCE = DECISIONS[EVIDENCE_COLUMNS].copy()
RUN_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
DECISIONS.to_parquet(LEDGER_PATH, index=False)
EVIDENCE.to_parquet(EVIDENCE_PATH, index=False)
print('Completed disease decisions:', f'{len(DECISIONS):,}/{len(EXPECTED_CACHE_KEYS):,}')
print('Evidence outcomes:', DECISIONS.decision.value_counts(dropna=False).to_dict())
print('Supported PMID evidence rows:', int(DECISIONS.decision.astype(str).eq('supported').sum()))
print('Saved run ledger:', LEDGER_PATH)
print('Saved pair evidence:', EVIDENCE_PATH)
LOG.info('Step 6 materialized decisions=%d evidence=%d missing=%d', len(DECISIONS), len(EVIDENCE), len(missing_keys))

2026-11-03 16:41 INFO Step 6 materialized decisions=38015 evidence=38015 missing=0


Completed disease decisions: 38,015/38,015
Evidence outcomes: {'unsupported': 32411, 'supported': 5604}
Supported PMID evidence rows: 5604
Saved run ledger: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/decision_ledger.parquet
Saved pair evidence: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/evidence/pathogen_host_evidence.parquet


## Step 6 · Review normalized outcomes and evidence

In [7]:
display(DECISIONS.groupby(['provenance_status', 'decision'], dropna=False).size().rename('records').to_frame())
supported_evidence = EVIDENCE.loc[EVIDENCE.decision.astype(str).eq('supported')]
print('Unique supported pathogen–host pairs:', supported_evidence[['pathogen_id', 'host_id']].drop_duplicates().shape[0])
print('Unique supporting PMIDs:', supported_evidence.pmid.astype(str).nunique())
print('Run evidence table:', EVIDENCE_PATH)
LOG.info('Step 6 review complete: %s', DECISIONS.groupby(['provenance_status', 'decision'], dropna=False).size().to_dict())

records
provenance_status decision            
system1_jev       supported       5604
                  unsupported    32411

2026-11-03 16:41 INFO Step 6 review complete: {('system1_jev', 'supported'): 5604, ('system1_jev', 'unsupported'): 32411}


Unique supported pathogen–host pairs: 348
Unique supporting PMIDs: 5268
Run evidence table: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/evidence/pathogen_host_evidence.parquet


The raw-response file and decision ledger are separate: the ledger is the normalized audit table; raw provider output is retained for review and reproducibility.